# AKAZI SCROLL - salary prediction Model 

# In this notebook, we will perform linear regression and compare four different models basing on LInkedin job postings dataset.

In [55]:
# load dataset

import pandas as pd

# Load the dataset

cols_needed = [
    'job_id', 'title', 'location', 'formatted_work_type',
    'formatted_experience_level', 'remote_allowed',
    'pay_period', 'min_salary', 'max_salary', 'med_salary',
    'normalized_salary', 'currency', 'compensation_type',
    'views', 'applies'
]

df = pd.read_csv("summative/linear_regression/data/postings.csv", usecols=cols_needed)
print("Shape:", df.shape)
df.head()
print(df.dtypes)
print()
print("Null counts:")
print(df.isnull().sum())
print()
print("Duplicate job_id count:", df['job_id'].duplicated().sum())
print("Fully duplicate rows:", df.duplicated().sum())



Shape: (123849, 15)
job_id                          int64
title                             str
max_salary                    float64
pay_period                        str
location                          str
views                         float64
med_salary                    float64
min_salary                    float64
formatted_work_type               str
applies                       float64
remote_allowed                float64
formatted_experience_level        str
currency                          str
compensation_type                 str
normalized_salary             float64
dtype: object

Null counts:
job_id                             0
title                              0
max_salary                     94056
pay_period                     87776
location                           0
views                           1689
med_salary                    117569
min_salary                     94056
formatted_work_type                0
applies                       100529
remote_allow

In [56]:
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
df_salary = df[df['normalized_salary'].notnull()].copy()
print("Rows with salary before filtering:", len(df_salary))
print(df_salary['normalized_salary'].describe())

Rows with salary before filtering: 36073
count        36,073.00
mean        205,327.04
std       5,097,626.76
min               0.00
25%          52,000.00
50%          81,500.00
75%         125,000.00
max     535,600,000.00
Name: normalized_salary, dtype: float64


In [57]:
LOWER_BOUND = 15_000
UPPER_BOUND = 400_000

before = len(df_salary)
df_salary = df_salary[(df_salary['normalized_salary'] >= LOWER_BOUND) & (df_salary['normalized_salary'] <= UPPER_BOUND)].copy()

after = len(df_salary)

print(f"Rows with salary before filtering: {before}")
print(f"Rows with salary after filtering: {after}")
print(f"Rows dropped as outliers: {before - after} ({100*(before-after)/before:.1f}%)")
print()
print(df_salary['normalized_salary'].describe())



Rows with salary before filtering: 36073
Rows with salary after filtering: 35449
Rows dropped as outliers: 624 (1.7%)

count    35,449.00
mean     95,044.30
std      54,194.51
min      15,000.00
25%      52,000.00
50%      82,500.00
75%     124,950.50
max     400,000.00
Name: normalized_salary, dtype: float64


In [58]:
feature_cols = [
    'title', 'location', 'formatted_work_type',
    'formatted_experience_level', 'remote_allowed',
    'views', 'applies'
]

print("Null counts in feature columns (salary-filtered subset):")
print(df_salary[feature_cols].isnull().sum())
print()
print("Unique value counts:")
for col in ['title', 'location', 'formatted_work_type', 'formatted_experience_level']:
    print(f"{col}: {df_salary[col].nunique()} unique values")

Null counts in feature columns (salary-filtered subset):
title                             0
location                          0
formatted_work_type               0
formatted_experience_level     7949
remote_allowed                30699
views                           479
applies                       26799
dtype: int64

Unique value counts:
title: 24104 unique values
location: 4490 unique values
formatted_work_type: 7 unique values
formatted_experience_level: 6 unique values


In [59]:

print("views nulls:", df_salary['views'].isnull().sum())
print("applies nulls:", df_salary['applies'].isnull().sum())
print()

leakage_cols = ['min_salary', 'max_salary', 'med_salary', 'pay_period', 'compensation_type', 'currency']
df_clean = df_salary.drop(columns=leakage_cols + ['remote_allowed'])

df_clean['formatted_experience_level'] = df_clean['formatted_experience_level'].fillna('Not Specified')

df_clean['views'] = df_clean['views'].fillna(0)
df_clean['applies'] = df_clean['applies'].fillna(0)

print("Final shape:", df_clean.shape)
print()
print("Remaining null counts:")
print(df_clean.isnull().sum())
print()
df_clean.head()

views nulls: 479
applies nulls: 26799

Final shape: (35449, 8)

Remaining null counts:
job_id                        0
title                         0
location                      0
views                         0
formatted_work_type           0
applies                       0
formatted_experience_level    0
normalized_salary             0
dtype: int64



,job_id,title,location,views,formatted_work_type,applies,formatted_experience_level,normalized_salary
0,921716,Marketing Coordinator,"Princeton, NJ",20.00,Full-time,2.00,Not Specified,"38,480.00"
1,1829192,Mental Health Therapist/Counselor,"Fort Collins, CO",1.00,Full-time,0.00,Not Specified,"83,200.00"
2,10998357,Assitant Restaurant Manager,"Cincinnati, OH",8.00,Full-time,0.00,Not Specified,"55,000.00"
3,23221523,Senior Elder Law / Trusts and Estates Associat...,"New Hyde Park, NY",16.00,Full-time,0.00,Not Specified,"157,500.00"
4,35982263,Service Technician,"Burlington, IA",3.00,Full-time,0.00,Not Specified,"70,000.00"


In [60]:
df_clean = df_clean.drop(columns=['applies'])

print("Final shape:", df_clean.shape)
df_clean.dtypes

Final shape: (35449, 7)


job_id                          int64
title                             str
location                          str
views                         float64
formatted_work_type               str
formatted_experience_level        str
normalized_salary             float64
dtype: object

In [61]:
def bucket_title(title):
    t = str(title).lower()
    if any(k in t for k in ['senior', 'sr.', 'sr ', 'lead', 'principal', 'staff']):
        return 'Senior/Lead'
    elif any(k in t for k in ['manager', 'director', 'head of', 'vp', 'chief', 'supervisor', 'superintendent']):
        return 'Management/Executive'
    elif any(k in t for k in ['engineer', 'developer', 'programmer', 'architect']):
        return 'Engineering'
    elif any(k in t for k in ['nurse', 'rn ', ' rn', 'cna', 'therapist', 'clinical', 'medical', 'health',
                               'physician', 'dental', 'phlebotomist', 'pathologist']):
        return 'Healthcare'
    elif any(k in t for k in ['sales', 'account executive', 'business development']):
        return 'Sales'
    elif any(k in t for k in ['marketing', 'seo', 'content', 'social media']):
        return 'Marketing'
    elif any(k in t for k in ['teacher', 'professor', 'instructor', 'tutor']):
        return 'Education'
    elif any(k in t for k in ['analyst', 'data scientist', 'data']):
        return 'Data/Analytics'
    elif any(k in t for k in ['accountant', 'finance', 'financial', 'auditor', 'accounts payable',
                               'accounts receivable', 'bookkeeper', 'payroll', 'controller',
                               'billing', 'accounting']):
        return 'Finance'
    elif any(k in t for k in ['designer', 'design']):
        return 'Design'
    elif any(k in t for k in ['hr ', 'human resources', 'recruiter', 'talent']):
        return 'HR/Recruiting'
    elif any(k in t for k in ['customer service', 'support']):
        return 'Customer Service/Support'
    elif any(k in t for k in ['technician', 'mechanic', 'operator', 'construction',
                               'material handler', 'warehouse', 'package handler']):
        return 'Technical/Trades'
    elif any(k in t for k in ['legal', 'attorney', 'paralegal']):
        return 'Legal'
    elif any(k in t for k in ['administrative', 'assistant', 'receptionist', 'coordinator']):
        return 'Administrative/Office Support'
    elif any(k in t for k in ['cashier', 'cook', 'housekeeper', 'team member', 'merchandiser',
                               'server', 'barista', 'retail']):
        return 'General Labor/Hospitality'
    else:
        return 'Other'

df_clean['title_category'] = df_clean['title'].apply(bucket_title)

print("Category distribution:")
print(df_clean['title_category'].value_counts())
print()
other_pct = (df_clean['title_category'] == 'Other').mean() * 100
print(f"'Other' now: {other_pct:.1f}%")
print(f"Reduced from {df_clean['title'].nunique()} unique titles to {df_clean['title_category'].nunique()} categories")

Category distribution:
title_category
Other                            7088
Management/Executive             6936
Senior/Lead                      5229
Healthcare                       2790
Engineering                      2751
Technical/Trades                 2149
Sales                            1731
Administrative/Office Support    1494
Data/Analytics                   1371
Finance                          1046
Customer Service/Support          653
General Labor/Hospitality         540
Legal                             526
Marketing                         336
HR/Recruiting                     300
Design                            265
Education                         244
Name: count, dtype: int64

'Other' now: 20.0%
Reduced from 24104 unique titles to 17 categories


In [62]:
df_clean = df_clean.drop(columns=['title'])
df_clean.head()

,job_id,location,views,formatted_work_type,formatted_experience_level,normalized_salary,title_category
0,921716,"Princeton, NJ",20.00,Full-time,Not Specified,"38,480.00",Marketing
1,1829192,"Fort Collins, CO",1.00,Full-time,Not Specified,"83,200.00",Healthcare
2,10998357,"Cincinnati, OH",8.00,Full-time,Not Specified,"55,000.00",Management/Executive
3,23221523,"New Hyde Park, NY",16.00,Full-time,Not Specified,"157,500.00",Senior/Lead
4,35982263,"Burlington, IA",3.00,Full-time,Not Specified,"70,000.00",Technical/Trades
